# 🎭 Deepfake Audio Detection Pipeline

This notebook demonstrates the end-to-end pipeline for detecting AI-generated speech. It loads an audio file, extracts features, loads our trained model, and makes a prediction.

In [ ]:
import sys
import os
from pathlib import Path
import torch
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt

# Add current directory to path so we can import our modules
sys.path.append(os.path.abspath('.'))

from src.model import build_model
from src.preprocessing import extract_features_from_audio
from src.utils import get_device

print(f"PyTorch Version: {torch.__version__}")
device = get_device()
print(f"Using device: {device}")


## 1. Load the Trained Model
We will load the `DeepfakeDetector` model from our best saved checkpoint.

In [ ]:
model_path = 'models/best_model.pth'

# Initialize model architecture
model = build_model(device=device, pretrained=False)

# Load weights
if os.path.exists(model_path):
    checkpoint = torch.load(model_path, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print("✅ Model weights loaded successfully!")
else:
    print(f"❌ Checkpoint not found at {model_path}. Please train the model first.")


## 2. Load and Visualize Audio
Let's test an audio file. We've copied some samples into the `apptest` directory.

In [ ]:
# You can change this to any valid path
audio_file = 'apptest/genuine_0003.wav'

if not os.path.exists(audio_file):
    # Fallback to a random file if specific one isn't found
    import glob
    files = glob.glob('apptest/*.wav')
    if files:
        audio_file = files[0]
        
print(f"Testing audio file: {audio_file}")

# Load audio to visualize waveform
y, sr = librosa.load(audio_file, sr=16000)

plt.figure(figsize=(10, 3))
librosa.display.waveshow(y, sr=sr, alpha=0.6)
plt.title("Audio Waveform")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.show()


## 3. Extract Features
Our pipeline uses Mel-Spectrogram and MFCC features concatenated into a 4-channel tensor.

In [ ]:
# Extract features using our preprocessing pipeline
features = extract_features_from_audio(audio_file)
print(f"Extracted feature tensor shape: {features.shape}")

# Visualize the first channel (Mel-Spectrogram)
plt.figure(figsize=(10, 4))
librosa.display.specshow(features[0].numpy(), sr=16000, x_axis='time', y_axis='mel')
plt.colorbar(format='%+2.0f dB')
plt.title('Mel-Spectrogram Channel')
plt.show()


## 4. Make a Prediction
Pass the features through the ResNet-BiLSTM model to get the Deepfake probability.

In [ ]:
# Prepare for model input (add batch dimension)
features = features.unsqueeze(0).to(device)

with torch.no_grad():
    # Forward pass
    logits = model(features)
    prob = torch.sigmoid(logits).item()

confidence = prob if prob >= 0.5 else 1.0 - prob
label = "Deepfake (AI-Generated)" if prob >= 0.5 else "Genuine (Human)"

print("="*40)
print(" PREDICTION RESULT")
print("="*40)
print(f" Classification : {label}")
print(f" Confidence     : {confidence * 100:.2f}%")
print(f" Deepfake Prob  : {prob:.4f}")
print("="*40)
